# Memory and conversation history with Pydantic AI

Pydantic AI manages conversation history through message lists. You can persist, filter, and summarize messages to maintain context across interactions.

```mermaid
flowchart LR
    User([User]) -->|Message| Agent
    Agent -->|Store| History["Message History"]
    History -->|Retrieve| Agent
    Agent -->|Response| User
```

Key concepts:
- **`all_messages()`** / **`new_messages()`**: Access full or current-run message history
- **`message_history`**: Pass previous messages to continue a conversation
- **`history_processors`**: Agent-level callables that filter/transform history before each LLM call

Reference: https://ai.pydantic.dev/message-history/

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessage

load_dotenv()

## Basic memory

Each agent run returns a result with `all_messages()` (includes prior history) and `new_messages()` (current run only). Pass messages back via `message_history` to continue a conversation.

**Note:** If `message_history` is set and not empty, a new system prompt is not generated — the existing history is assumed to include one.

In [ ]:
agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant.",
)

# First turn
result1 = agent.run_sync("Hi! My name is Dylan.")
print("Agent:", result1.output)

# Second turn - pass message history to maintain context
result2 = agent.run_sync("What's my name?", message_history=result1.all_messages())
print("Agent:", result2.output)

## Inspecting message history

You can inspect the full message history to see exactly what was sent and received.

In [ ]:
for msg in result2.all_messages():
    print(type(msg).__name__, "-", msg)

## Conversation loop

Build a multi-turn conversation by accumulating message history across runs.

In [ ]:
agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Keep your responses concise.",
)

message_history: list[ModelMessage] = []

conversations = [
    "Hi! I'm Dylan.",
    "I like to play tennis.",
    "Carlos Alcaraz is an amazing tennis player, isn't he?",
    "What's my name and what sport do I like?",
]

for user_msg in conversations:
    print(f"User: {user_msg}")
    result = agent.run_sync(user_msg, message_history=message_history)
    print(f"Agent: {result.output}")
    print()
    message_history = result.all_messages()

## Serializing message history

For persistence (e.g., saving to a database), use `ModelMessagesTypeAdapter` to serialize/deserialize messages.

In [ ]:
from pydantic_core import to_jsonable_python
from pydantic_ai import ModelMessagesTypeAdapter

# Serialize to JSON-compatible Python objects
history = result.all_messages()
as_python = to_jsonable_python(history)
print(f"Serialized {len(as_python)} messages")

# Restore from serialized form
restored = ModelMessagesTypeAdapter.validate_python(as_python)

# Use restored history in a new run
result3 = agent.run_sync("Remind me, what's my name?", message_history=restored)
print(f"Agent: {result3.output}")

## History Processors: Keep recent messages

Agents accept a `history_processors` parameter — a list of callables that take a list of `ModelMessage` and return a modified list. Processors run before each LLM call.

```mermaid
flowchart LR
    History["Full History"] --> P1["Processor 1"]
    P1 --> P2["Processor 2"]
    P2 --> LLM["LLM Call"]
```

**Important:** When slicing history, ensure tool calls and returns remain paired, otherwise the LLM may return an error.

In [ ]:
def keep_recent_messages(
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    """Keep only the last 5 messages."""
    if len(messages) > 5:
        return messages[-5:]
    return messages


agent_with_trim = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Keep your responses concise.",
    history_processors=[keep_recent_messages],
)

message_history: list[ModelMessage] = []

conversations = [
    "Hi! I'm Dylan.",
    "I like to play tennis.",
    "Carlos Alcaraz is an amazing tennis player.",
    "I also enjoy cooking Italian food.",
    "What's my name?",
]

for user_msg in conversations:
    print(f"User: {user_msg}")
    result = agent_with_trim.run_sync(user_msg, message_history=message_history)
    print(f"Agent: {result.output}")
    print(f"  (History size: {len(result.all_messages())} messages)")
    print()
    message_history = result.all_messages()

## History Processors: Context-aware processing

Processors can accept a `RunContext` as the first argument to make decisions based on runtime state like token usage.

In [ ]:
from pydantic_ai import RunContext


def context_aware_processor(
    ctx: RunContext[None],
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    """Trim more aggressively when token usage is high."""
    if ctx.usage.total_tokens and ctx.usage.total_tokens > 1000:
        print(f"  [Trimming: {ctx.usage.total_tokens} tokens used]")
        return messages[-3:]
    return messages


agent_ctx = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Keep your responses concise.",
    history_processors=[context_aware_processor],
)

message_history: list[ModelMessage] = []

conversations = [
    "Hi! I'm Dylan.",
    "Tell me about the history of tennis in detail.",
    "Now tell me about the top 10 tennis players of all time.",
    "What was my name again?",
]

for user_msg in conversations:
    print(f"User: {user_msg}")
    result = agent_ctx.run_sync(user_msg, message_history=message_history)
    print(f"Agent: {result.output}")
    print()
    message_history = result.all_messages()

## History Processors: Summarization

Instead of discarding old messages, you can summarize them and keep only the summary plus recent messages.

In [ ]:
summarizer = Agent(
    "openai:gpt-5-mini",
    system_prompt="Create a brief summary of the conversation so far. Focus on key facts mentioned by the user.",
)


async def summarize_old_messages(
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    """Summarize old messages when history gets too long."""
    if len(messages) > 8:
        # Summarize the oldest messages
        oldest = messages[:-2]
        summary_result = await summarizer.run(
            "Summarize this conversation", message_history=oldest
        )
        print(f"  [Summarized: {summary_result.output[:80]}...]")
        # Return summary as new messages + the most recent exchange
        return summary_result.new_messages() + messages[-2:]
    return messages


agent_summarize = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Keep your responses concise.",
    history_processors=[summarize_old_messages],
)

message_history: list[ModelMessage] = []

conversations = [
    "Hi! I'm Dylan.",
    "I like to play tennis.",
    "Carlos Alcaraz is amazing.",
    "I also enjoy cooking Italian food.",
    "My favorite color is blue.",
    "What's my name and what are my hobbies?",
]

for user_msg in conversations:
    print(f"User: {user_msg}")
    result = await agent_summarize.run(user_msg, message_history=message_history)
    print(f"Agent: {result.output}")
    print()
    message_history = result.all_messages()

## Chaining multiple processors

Processors apply sequentially. You can combine multiple strategies.

In [ ]:
def log_history_size(
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    """Log the history size (useful for debugging)."""
    print(f"  [History: {len(messages)} messages]")
    return messages


agent_chained = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Keep your responses concise.",
    history_processors=[log_history_size, keep_recent_messages],
)

message_history: list[ModelMessage] = []

for user_msg in ["Hi, I'm Dylan.", "I like tennis.", "Who am I?"]:
    print(f"User: {user_msg}")
    result = agent_chained.run_sync(user_msg, message_history=message_history)
    print(f"Agent: {result.output}\n")
    message_history = result.all_messages()

## Exercise

Build a multi-turn chatbot that remembers user preferences (favorite color, food, hobby) and can recall them later. Implement both a trimming and a summarization `history_processor` and compare the results.

In [ ]:
preference_turns = [
    "My favorite color is green.",
    "My favorite food is sushi.",
    "My hobby is photography.",
    "I also like weekend hikes.",
    "Can you remind me of my favorite color, food, and hobby?",
]


def keep_recent_preferences(
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    return messages[-6:]


preference_summarizer = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "Summarize the user's stable preferences. Keep favorite color, "
        "favorite food, hobbies, and any other enduring facts."
    ),
)


async def summarize_preferences(
    messages: list[ModelMessage],
) -> list[ModelMessage]:
    if len(messages) > 6:
        summary = await preference_summarizer.run(
            "Summarize the user's long-term preferences from this conversation.",
            message_history=messages[:-2],
        )
        return summary.new_messages() + messages[-2:]
    return messages


trimmed_preference_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are a helpful assistant. Remember user preferences and answer concisely."
    ),
    history_processors=[keep_recent_preferences],
)

summarized_preference_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are a helpful assistant. Remember user preferences and answer concisely."
    ),
    history_processors=[summarize_preferences],
)


def run_trimmed_chat(turns: list[str]) -> str:
    history: list[ModelMessage] = []
    final_answer = ""
    for turn in turns:
        result = trimmed_preference_agent.run_sync(
            turn,
            message_history=history,
        )
        history = result.all_messages()
        final_answer = result.output
    return final_answer


async def run_summarized_chat(turns: list[str]) -> str:
    history: list[ModelMessage] = []
    final_answer = ""
    for turn in turns:
        result = await summarized_preference_agent.run(
            turn,
            message_history=history,
        )
        history = result.all_messages()
        final_answer = result.output
    return final_answer

In [ ]:
trimmed_answer = run_trimmed_chat(preference_turns)
summarized_answer = await run_summarized_chat(preference_turns)

print("Trimmed history answer:")
print(trimmed_answer)
print("\nSummarized history answer:")
print(summarized_answer)